<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


**Method: Random Forest classifier**

 each page is labeled `up`, `stable`, or `down` based on its month-over-month change in impressions (±20% threshold). The task is therefore classification.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


**Split: grouped by client, stratified by class**

 All pages of a client go entirely to train or test, because pages of the same client share keywords and site quality, so a page-level split would leak information and inflate the scores. The split is a single train/test split made with `GroupShuffleSplit`, repeated with a different seed until each class (`up`, `stable`, `down`) makes up at least 10% of both sets.



In [15]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

# ---------- الاتصال ----------
TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"

# ---------- الإعدادات ----------
FEAT_START, FEAT_END   = "2026-03-01", "2026-03-31"   # الـ features: شهر 3
MID                    = "2026-03-15"                 # آخر يوم في النص الأول
LABEL_START, LABEL_END = "2026-04-01", "2026-04-30"   # الـ label: شهر 4
MIN_IMPR = 100                                        # الحد الأدنى للظهور في شهر 3

# ---------- الصفحات (content_slice) ----------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE content_slice AS
SELECT d.client_hash_id, d.content_hash_id
FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
JOIN read_parquet('{SRC_CLIENTS}') c USING (client_hash_id)
WHERE d.month = '2026-03'
  AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
  AND CAST(c.gsc_data_start AS DATE) < DATE '{FEAT_START}'
GROUP BY d.client_hash_id, d.content_hash_id
HAVING SUM(d.gsc_impressions) >= {MIN_IMPR}
""")
print("pages:", con.execute("SELECT COUNT(*) FROM content_slice").fetchone()[0])

# ---------- الداتا: 14 feature + label ----------
con.execute(f"""
CREATE OR REPLACE TEMP TABLE dataset AS
WITH daily AS (
    SELECT d.client_hash_id, d.content_hash_id, d.report_date,
           d.gsc_impressions, d.gsc_clicks, d.gsc_sum_position,
           d.ga4_pageviews, d.ga4_sessions, d.ga4_users,
           d.ga4_engaged_sessions, d.ga4_total_engagement_sec,
           COALESCE(d.report_date >= CAST(c.ga4_data_start AS DATE), FALSE) AS ga4_ok
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN content_slice s
      ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
    JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
    WHERE d.month IN ('2026-03','2026-04')
      AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
      AND d.report_date >= CAST(c.gsc_data_start AS DATE)
),
agg AS (
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                         AS f01,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)                 AS f02,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)           AS f05,
      SUM(CASE WHEN ga4_ok THEN ga4_engaged_sessions END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f06,
      SUM(CASE WHEN ga4_ok THEN ga4_total_engagement_sec END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f07,
      SUM(CASE WHEN ga4_ok THEN ga4_pageviews END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f08,
      SUM(CASE WHEN ga4_ok THEN ga4_users END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f09
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
),
halves AS (
    SELECT client_hash_id, content_hash_id,
      (report_date > DATE '{MID}') AS is_h2,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                AS avg_impr,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)        AS ctr,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS pos
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) > 0
),
trend AS (
    SELECT client_hash_id, content_hash_id,
      (MAX(CASE WHEN is_h2 THEN avg_impr END) - MAX(CASE WHEN NOT is_h2 THEN avg_impr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN avg_impr END), 0)       AS f03,
      (MAX(CASE WHEN is_h2 THEN ctr END) - MAX(CASE WHEN NOT is_h2 THEN ctr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN ctr END), 0)            AS f04,
      MAX(CASE WHEN is_h2 THEN pos END) - MAX(CASE WHEN NOT is_h2 THEN pos END) AS f14
    FROM halves
    GROUP BY 1, 2
),
dim AS (
    SELECT content_hash_id,
      DATE_DIFF('day',
        CAST(CASE WHEN content_updated_date IS NOT NULL
                   AND content_updated_date <= DATE '{FEAT_END}'
                  THEN content_updated_date ELSE content_created_date END AS DATE),
        DATE '{FEAT_END}')                                                   AS f10,
      CASE WHEN keyword_hash_id IS NULL OR keyword_hash_id = '' THEN NULL
           ELSE COUNT(*) OVER (PARTITION BY client_hash_id, keyword_hash_id) - 1
      END                                                                    AS f11,
      CASE WHEN main_intent IS NULL THEN NULL
           ELSE DENSE_RANK() OVER (ORDER BY main_intent) - 1 END             AS f12,
      backlinks                                                              AS f13
    FROM read_parquet('{SRC_CONTENT}')
),
lab AS (
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN report_date END), 0)                                     AS avg_base,
      COALESCE(SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN report_date END), 0), 0)                                 AS avg_next
    FROM daily
    GROUP BY 1, 2
)
SELECT s.content_hash_id AS content_id,
       s.client_hash_id  AS client_id,
       a.f01 AS avg_daily_impressions,
       a.f02 AS ctr,
       t.f03 AS impressions_trend,
       t.f04 AS ctr_trend,
       a.f05 AS weighted_avg_position,
       a.f06 AS engagement_rate,
       a.f07 AS engagement_time_per_session,
       a.f08 AS avg_daily_pageviews,
       a.f09 AS avg_daily_users,
       m.f10 AS page_age_days,
       m.f11 AS same_keyword_pages,
       m.f12 AS main_intent_code,
       m.f13 AS backlinks,
       t.f14 AS position_trend,
       CASE WHEN l.avg_next > 1.2 * l.avg_base THEN 'up'
            WHEN l.avg_next < 0.8 * l.avg_base THEN 'down'
            ELSE 'stable' END AS label
FROM content_slice s
LEFT JOIN agg   a ON a.client_hash_id = s.client_hash_id AND a.content_hash_id = s.content_hash_id
LEFT JOIN trend t ON t.client_hash_id = s.client_hash_id AND t.content_hash_id = s.content_hash_id
LEFT JOIN lab   l ON l.client_hash_id = s.client_hash_id AND l.content_hash_id = s.content_hash_id
LEFT JOIN dim   m ON m.content_hash_id = s.content_hash_id
""")

df = con.execute("SELECT * FROM dataset").df()
print(df.shape)
df

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 101168


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101168, 17)


,content_id,client_id,avg_daily_impressions,ctr,impressions_trend,ctr_trend,weighted_avg_position,engagement_rate,engagement_time_per_session,avg_daily_pageviews,avg_daily_users,page_age_days,same_keyword_pages,main_intent_code,backlinks,position_trend,label
0,content_49c3b07c220bb3b6,client_73cda7b4e4f265ea,9.258065,0.000000,0.304688,NaN,14.923345,NaN,NaN,0.0,0.0,34,0,1,<NA>,-1.406337,up
1,content_008eebad20448a91,client_73cda7b4e4f265ea,52.677419,0.003674,-0.092997,-0.793275,3.968769,NaN,NaN,0.0,0.0,49,0,1,0,-1.166079,stable
2,content_cf5de5bef40443cb,client_73cda7b4e4f265ea,19.032258,0.001695,-0.558136,-1.000000,14.020339,NaN,NaN,0.0,0.0,34,0,1,0,12.519125,down
3,content_c6d1c0f0f2a76472,client_73cda7b4e4f265ea,3.806452,0.000000,-0.401042,NaN,1.237288,NaN,NaN,0.0,0.0,34,0,1,0,0.003019,up
4,content_e26729b0737b54eb,client_73cda7b4e4f265ea,88.354839,0.001460,-0.024078,1.881890,4.447974,0.0,1.0,0.5,0.5,371,0,1,<NA>,-0.033338,up
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101163,content_c0ac65cbca8d0b84,client_08a6a72ff48e62c0,6.709677,0.000000,NaN,NaN,22.740385,NaN,NaN,NaN,NaN,349,0,1,<NA>,NaN,up
101164,content_6f0a484a0bce67a2,client_08a6a72ff48e62c0,9.645161,0.006689,5.848958,-0.863118,9.163880,NaN,NaN,NaN,NaN,48,0,1,319,-2.182193,up
101165,content_6f1abfe832d6e6be,client_08a6a72ff48e62c0,66.000000,0.000489,0.279986,-1.000000,8.990225,NaN,NaN,NaN,NaN,343,0,1,<NA>,-0.990300,up
101166,content_5aed6ebee45fbcb0,client_08a6a72ff48e62c0,5.290323,0.000000,13.437500,NaN,2.457317,NaN,NaN,NaN,NaN,110,0,1,0,-3.879221,up


In [18]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

TEST_PAGES = 0.20     # نسبة الصفحات في الـ test
PAGES_TOL  = 0.02     # المدى المقبول: 18% لـ 22%
MIN_SHARE  = 0.20     # أقل نسبة لكل class (لو منفعتش نزّلها لـ 0.15 ثم 0.10)
MAX_TRIES  = 2000
MIN_TEST_CLIENTS = 5      # أقل عدد كلاينتس في الـ test

def split_by_client(df, label_col="label", group_col="client_id",
                    test_pages=TEST_PAGES, tol=PAGES_TOL,
                    min_share=MIN_SHARE, max_tries=MAX_TRIES,
                    min_test_clients=MIN_TEST_CLIENTS):
    clients = np.sort(df[group_col].unique())
    classes = df[label_col].unique()

    for seed in range(max_tries):
        rng = np.random.default_rng(seed)
        shuffled = rng.permutation(clients)
        sizes = df[group_col].value_counts()

        # اسحب كلاينتس للـ test لحد ما نوصل لـ 20% من الصفحات
        target = test_pages * len(df)
        test_clients, total = [], 0
        for c in shuffled:
            if total >= target:
                break
            test_clients.append(c)
            total += sizes[c]

        is_test = df[group_col].isin(test_clients)
        train, test = df[~is_test], df[is_test]

        if abs(len(test) / len(df) - test_pages) > tol:
            continue
        if test[group_col].nunique() < min_test_clients:
            continue

        tr_share = train[label_col].value_counts(normalize=True)
        te_share = test[label_col].value_counts(normalize=True)

        if all(tr_share.get(c, 0) >= min_share and te_share.get(c, 0) >= min_share
               for c in classes):
            return train.copy(), test.copy(), seed

    raise RuntimeError(f"No valid split in {max_tries} seeds")


# الاستخدام (df = slice_df + عمود label)
train_df, test_df, seed = split_by_client(df)

print("seed:", seed)
print("train:", len(train_df), "pages |", train_df["client_id"].nunique(), "clients")
print("test: ", len(test_df), "pages |", test_df["client_id"].nunique(), "clients")
print("\ntrain label share:\n", train_df["label"].value_counts(normalize=True).round(3))
print("\ntest label share:\n", test_df["label"].value_counts(normalize=True).round(3))

# تأكيد إن مفيش كلاينت في الجهتين
assert set(train_df["client_id"]).isdisjoint(test_df["client_id"])

seed: 2
train: 79821 pages | 26 clients
test:  21347 pages | 13 clients

train label share:
 label
down      0.534
up        0.234
stable    0.232
Name: proportion, dtype: float64

test label share:
 label
down      0.390
up        0.309
stable    0.302
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

FEATURES = [
    "avg_daily_impressions", "ctr", "impressions_trend", "ctr_trend",
    "weighted_avg_position", "engagement_rate", "engagement_time_per_session",
    "avg_daily_pageviews", "avg_daily_users", "page_age_days",
    "same_keyword_pages", "main_intent_code", "backlinks", "position_trend",
]

X_train, y_train = train_df[FEATURES], train_df["label"]
X_test,  y_test  = test_df[FEATURES],  test_df["label"]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=30,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print("test accuracy:", round(accuracy_score(y_test, pred), 4))

test accuracy: 0.5045


In [20]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd

labels = ["down", "stable", "up"]

print("macro F1:", round(f1_score(y_test, pred, average="macro"), 4))
print()
print(classification_report(y_test, pred, labels=labels, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, pred, labels=labels),
    index=[f"true_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)
print(cm)

macro F1: 0.49

              precision    recall  f1-score   support

        down      0.522     0.645     0.577      8318
      stable      0.415     0.340     0.374      6438
          up      0.555     0.488     0.519      6591

    accuracy                          0.505     21347
   macro avg      0.497     0.491     0.490     21347
weighted avg      0.500     0.505     0.498     21347

             pred_down  pred_stable  pred_up
true_down         5368         1684     1266
true_stable       2935         2186     1317
true_up           1978         1397     3216


In [25]:
import numpy as np

K = 500
CLS = "down"   # غيّرها لـ "up" أو "stable" لو عايز

proba = rf.predict_proba(X_test)
col = list(rf.classes_).index(CLS)

ranked = test_df[["content_id", "client_id", "label"]].copy()
ranked["confidence"] = proba[:, col]
ranked = ranked.sort_values("confidence", ascending=False).reset_index(drop=True)

top = ranked.head(K)
precision_k = (top["label"] == CLS).mean()

print(f"class: {CLS}")
print(f"precision@{K}: {precision_k:.3f}")
print(f"confidence range in top {K}: {top['confidence'].min():.3f} -> {top['confidence'].max():.3f}")
print(f"base rate of {CLS} in test: {(ranked['label'] == CLS).mean():.3f}")



class: down
precision@500: 0.546
confidence range in top 500: 0.777 -> 0.933
base rate of down in test: 0.390


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.